# 1. Answer the questions from the Preamble and Introduction

1. Проклятие размерности актуально для KNN, так как при увеличении размерности расстояния между точками становятся почти одинаковыми, и соседи перестают быть действительно близкими. Для линейных моделей, таких как Ridge, Lasso и логистическая регрессия, проклятие размерности проявляется в том, что требуется экспоненциально больше данных для стабильного обучения, иначе модель переобучается. Random Forest и градиентный бустинг также подвержены этой проблеме, хотя и в меньшей степени, так как при очень большом количестве признаков качество снижается из-за разреженности данных. Методы на основе расстояний, такие как SVM с ядрами и кластеризация, также страдают из-за того, что расстояния перестают быть информативными.

**2. В чём разница между PCA и SVD.**

PCA — это метод, который ищет направления, по которым данные больше всего разбросаны. Он берет ковариационную матрицу и находит ее собственные векторы — это и есть главные компоненты. SVD — это просто способ разложить любую матрицу на три: U, сигма и V в степени T. SVD не надо считать ковариационную матрицу, он работает с исходными данными. На практике PCA часто делают через SVD, потому что так быстрее и устойчивее, особенно если данных много.

**3. В чём разница между NMF и SVD.**

NMF — это разложение на две неотрицательные матрицы: исходная приблизительно равна W * H. Здесь важно, что все числа должны быть неотрицательными. Получается, что исходные данные собираются из "кирпичиков" (компонент), которые складываются друг с другом. Это удобно для интерпретации, например, в текстах или изображениях. SVD же не требует неотрицательности, его компоненты могут быть отрицательными, и они ортогональны. Из-за этого интерпретировать их сложнее. Главное отличие: NMF дает только плюсы и работает с аддитивными компонентами, а SVD дает и плюсы, и минусы, и его компоненты ортогональны.

**4. Опишите структуру алгоритма уменьшения размерности локально линейного вложения (LLE).**

LLE работает в три этапа.

Сначала для каждой точки находим K ближайших соседей — просто берем самые близкие точки по евклидову расстоянию.

Потом для каждой точки подбираем такие веса, чтобы через соседей можно было восстановить саму точку. То есть мы хотим представить точку как взвешенную сумму соседей. Веса подбираются так, чтобы ошибка была минимальной, и чтобы они в сумме давали единицу.

И наконец, мы ищем новые координаты точек в пространстве меньшей размерности, но так, чтобы те же самые веса как можно лучше восстанавливали точки. Получается, что локальные связи между точками сохраняются. Задача сводится к поиску собственных векторов одной специальной матрицы.

# 2. Classification with Sparse Features

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA, TruncatedSVD
import umap.umap_ as umap
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE, LocallyLinearEmbedding

In [ ]:
ratings = pd.read_csv('data/Ratings.csv')
users = pd.read_csv('data/Users.csv')
ratings

In [ ]:
users

In [ ]:
data = ratings.merge(users, on='User-ID')
data = data[data['Age'].notna()]

In [ ]:
user_ids = data['User-ID'].unique()
book_ids = data['ISBN'].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
book_to_idx = {b: i for i, b in enumerate(book_ids)}

rows = data['User-ID'].map(user_to_idx)
cols = data['ISBN'].map(book_to_idx)
vals = data['Book-Rating']

X = csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(book_ids)))
y = data.groupby('User-ID')['Age'].first().values

print(f"Разреженная матрица: {X.shape}, y: {y.shape}")

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X   , y, test_size=0.4, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
ridge = Ridge(random_state=42)
params_ridge = {'alpha': [1, 10, 50, 100, 200], 'solver': ['auto', 'lsqr', 'sparse_cg']}
rs_ridge = RandomizedSearchCV(ridge, params_ridge, n_iter=6, cv=3, scoring='r2', n_jobs=-1, random_state=42)

start = time.time()
rs_ridge.fit(X_train, y_train)
time_ridge = round(time.time()-start,2)
r2_ridge = r2_score(y_valid, rs_ridge.predict(X_valid))

In [ ]:
rf = RandomForestRegressor(random_state=42)
params_rf = {'n_estimators':[20,40], 'max_depth':[5,10], 'min_samples_split':[2,5]}
rs_rf = RandomizedSearchCV(rf, params_rf, n_iter=4, cv=3, scoring='r2', n_jobs=-1, random_state=42)
start = time.time()
rs_rf.fit(X_train, y_train)
time_rf = round(time.time()-start,2)
r2_rf = r2_score(y_valid, rs_rf.predict(X_valid))

In [ ]:
print(f"Ridge R² на валидации: {r2_ridge:.4f}, время: {time_ridge} сек")
print(f"Random Forest R² на валидации: {r2_rf:.4f}, время: {time_rf} сек")


In [ ]:
print(X_train.shape, X_valid.shape, X_test.shape)

In [ ]:
pca = TruncatedSVD(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_valid_pca = pca.transform(X_valid)
X_test_pca = pca.transform(X_test)

In [ ]:
print(X_train_pca.shape, X_valid_pca.shape, X_test_pca.shape)

In [ ]:
reducer = umap.UMAP(n_components=100, random_state=42)
X_train_umap = reducer.fit_transform(X_train)
X_valid_umap = reducer.transform(X_valid)
X_test_umap = reducer.transform(X_test)

In [ ]:
print(X_train_umap.shape, X_valid_umap.shape, X_test_umap.shape)

In [ ]:
ridge_pca = Ridge(random_state=42)
params_ridge = {'alpha': [1, 10, 50, 100, 200]}
rs_ridge_pca = RandomizedSearchCV(ridge_pca, params_ridge, n_iter=5, cv=3, scoring='r2', random_state=42)

start = time.time()
rs_ridge_pca.fit(X_train_pca, y_train)
time_ridge_pca = round(time.time() - start, 2)
r2_ridge_pca = r2_score(y_valid, rs_ridge_pca.predict(X_valid_pca))
print(f"Ridge на PCA: R² = {r2_ridge_pca:.4f}, время = {time_ridge_pca} сек")

In [ ]:
rf_pca = RandomForestRegressor(random_state=42)
params_rf = {'n_estimators': [20, 40], 'max_depth': [5, 10]}
rs_rf_pca = RandomizedSearchCV(rf_pca, params_rf, n_iter=4, cv=3, scoring='r2', random_state=42)

start = time.time()
rs_rf_pca.fit(X_train_pca, y_train)
time_rf_pca = round(time.time() - start, 2)
r2_rf_pca = r2_score(y_valid, rs_rf_pca.predict(X_valid_pca))
print(f"Random Forest на PCA: R² = {r2_rf_pca:.4f}, время   = {time_rf_pca} сек")

In [ ]:
ridge_umap = Ridge(random_state=42)
rs_ridge_umap = RandomizedSearchCV(ridge_umap, params_ridge, n_iter=5, cv=3, scoring='r2', random_state=42)

start = time.time()
rs_ridge_umap.fit(X_train_umap, y_train)
time_ridge_umap = round(time.time() - start, 2)
r2_ridge_umap = r2_score(y_valid, rs_ridge_umap.predict(X_valid_umap))
print(f"Ridge на UMAP: R² = {r2_ridge_umap:.4f}, время = {time_ridge_umap} сек")

In [ ]:
rf_umap = RandomForestRegressor(random_state=42)
rs_rf_umap = RandomizedSearchCV(rf_umap, params_rf, n_iter=4, cv=3, scoring='r2', random_state=42)

start = time.time()
rs_rf_umap.fit(X_train_umap, y_train)
time_rf_umap = round(time.time() - start, 2)
r2_rf_umap = r2_score(y_valid, rs_rf_umap.predict(X_valid_umap))
print(f"Random Forest на UMAP: R² = {r2_rf_umap:.4f}, время = {time_rf_umap} сек")

In [ ]:
print(f"Ridge (исходные)          {time_ridge}с         R2={r2_ridge:.4f}")
print(f"Random Forest (исходные)  {time_rf}с           R2={r2_rf:.4f}")
print(f"Ridge + PCA               {time_ridge_pca}с           R2={r2_ridge_pca:.4f}")
print(f"Random Forest + PCA       {time_rf_pca}с          R2= {r2_rf_pca:.4f}")
print(f"Ridge + UMAP              {time_ridge_umap}с          R2= {r2_ridge_umap:.4f}")
print(f"Random Forest + UMAP      {time_rf_umap}с          R2= {r2_rf_umap:.4f}")

Метрики плохие

## 3. Визуализации

In [ ]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1)
X = mnist['data'].values
y = mnist['target'].values

In [ ]:
image_digit = X[0].reshape(28, 28)
print(image_digit.shape)
vector_digit = image_digit.flatten()
print(vector_digit.shape)
plt.imshow(image_digit)

In [ ]:
def scatter_2d(X, y, title):
    plt.figure(figsize=(15, 8))
    sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, s=100, palette='tab10', alpha=0.3)
    plt.title(f"{title} на MNIST (2D)")
    plt.xlabel("Компонента 1")
    plt.ylabel("Компонента 2")
    plt.legend(title='Цифры', loc='upper right') 
    plt.show()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
scatter_2d(X_pca, y, 'PCA')

In [ ]:
svd = TruncatedSVD(n_components=2)
X_svd = svd.fit_transform(X)
scatter_2d(X_svd, y, 'SVD')

In [ ]:
tsne = TSNE(n_components=2)
X_tsne = tsne.fit_transform(X)
scatter_2d(X_tsne, y, 'TSNE')

In [ ]:
umap = umap.UMAP(n_components=2)
X_umap = umap.fit_transform(X)
scatter_2d(X_umap, y, 'UMAP')

In [ ]:
lle = LocallyLinearEmbedding(n_components=2)
X_lle = lle.fit_transform(X[:20000])
scatter_2d(X_lle, y[:20000], 'LLE')

**Метрики для оценки:**

- **Silhouette Score** — насколько хорошо сформированы кластеры (от -1 до 1, чем выше, тем лучше)
- **Calinski-Harabasz Index** — отношение межкластерной дисперсии к внутрикластерной (чем выше, тем лучше)
- **Davies-Bouldin Index** — средняя "похожесть" кластеров друг на друга (чем ниже, тем лучше)
- **KNN Accuracy** — точность классификации KNN на спроецированных данных (чем выше, тем лучше)


In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

def evaluate(X_emb, y, name):
    sil = silhouette_score(X_emb, y)
    cal = calinski_harabasz_score(X_emb, y)
    db = davies_bouldin_score(X_emb, y)

    knn = KNeighborsClassifier(n_neighbors=5)
    acc = cross_val_score(knn, X_emb, y, cv=3).mean()

    print(f"{name}:")
    print(f"  Silhouette = {sil:.4f}")
    print(f"  Calinski-Harabasz = {cal:.2f}")
    print(f"  Davies-Bouldin = {db:.4f}")
    print(f"  KNN Accuracy = {acc:.4f}")
    print()

In [ ]:
evaluate(X_pca, y, "PCA")
evaluate(X_svd, y, "SVD")
evaluate(X_umap, y, "UMAP")

evaluate(X_tsne, y, "t-SNE")
evaluate(X_lle, y[:20000], "LLE")

По силуэту UMAP лучше всех — 0.52, за ним t-SNE с 0.38, потом LLE, а PCA и SVD вообще около нуля. Это значит, что в UMAP классы получились самыми четкими и отделенными друг от друга.

По индексу Калински-Харабаса тоже UMAP лидирует — 108405, дальше t-SNE, LLE и с большим отрывом PCA с SVD. Чем выше этот показатель, тем лучше кластеры отделены друг от друга.

По индексу Дэвиса-Болдина (чем ниже, тем лучше) UMAP снова первый — 0.72, у t-SNE 0.93, у LLE 2.63, а у PCA и SVD вообще 5.9 и 10.7. Это подтверждает, что UMAP и t-SNE почти не перемешивают классы между собой.

По точности KNN лучше всех t-SNE — 0.97, у UMAP 0.96, у LLE 0.83, а PCA и SVD показали 0.42 и 0.30.

Получается, что по трем метрикам из четырех UMAP впереди, а по точности KNN чуть лучше t-SNE. Оба метода очень хороши. LLE работает средне. PCA и SVD вообще не справились с задачей.

# 3. Сжатие изображений с использованием SVD



In [ ]:
from skimage import data
from skimage.transform import resize
import random

In [ ]:
random.seed(42)
all_images = [
    data.camera(),
    data.text(),
    data.coins(),
    data.clock(),
    data.page(),
    data.moon(),
    data.checkerboard(),
    data.brick(),
    data.grass(),
    data.rocket()
]

selected = random.sample(all_images, 3)
images = [resize(img, (256, 256)) for img in selected]

plt.figure(figsize=(12, 4))
for i, img in enumerate(images):
    plt.subplot(1, 3, i+1)
    plt.imshow(img, cmap='gray')
    plt.title(f'Изображение {i+1}')
    plt.axis('off')
plt.suptitle('Случайные 3 изображения для SVD')
plt.show()
names = ['Изображение 1', 'Изображение 2', 'Изображение 3']

In [ ]:
ranks = [1, 2, 5, 10, 20, 30, 50, 75, 100, 150]

plt.figure(figsize=(20, 8))
for i, (img, name) in enumerate(zip(images, names)):
    U, s, Vt = np.linalg.svd(img, full_matrices=False)
    
    for j, r in enumerate(ranks):
        plt.subplot(3, 10, i*10 + j + 1)
        img_approx = U[:, :r] @ np.diag(s[:r]) @ Vt[:r, :]
        plt.imshow(img_approx, cmap='gray')
        plt.title(f'ранг={r}')
        plt.axis('off')
    print(f"{name}: первые сингулярные значения: {s[:5]}")

plt.suptitle('Низкоранговые приближения изображений')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(15, 4))
for i, (img, name) in enumerate(zip(images, names)):
    U, s, Vt = np.linalg.svd(img, full_matrices=False)
    plt.subplot(1, 3, i+1)
    plt.plot(s[:50], 'bo-', markersize=3)
    plt.title(f'Спектр СЗ - {name}')
    plt.xlabel('Номер')
    plt.ylabel('Значение')
    plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 4))
for i, (img, name) in enumerate(zip(images, names)):
    U, s, Vt = np.linalg.svd(img, full_matrices=False)
    explained_var = (s**2) / np.sum(s**2)
    cumulative_var = np.cumsum(explained_var)
    
    plt.subplot(1, 3, i+1)
    plt.plot(cumulative_var[:100], 'r-', linewidth=2)
    plt.title(f'Объясненная дисперсия - {name}')
    plt.xlabel('Ранг')
    plt.ylabel('Накопленная дисперсия')
    plt.grid(True)
    plt.legend()
plt.tight_layout()
plt.show()

# 5. Обнаружение фона с помощью SVD

In [ ]:

import numpy as np
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture("data/Video_008.avi")
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, (160, 120))
    frames.append(gray.flatten())
cap.release()

X = np.array(frames).T
h, w = 120, 160
U, s, Vt = np.linalg.svd(X, full_matrices=False)

max_rank = min(X.shape)
ranks = [1, 2, 3, 5, 10, 20, 30, 50, 100, 200, 500, 1000, max_rank]
ranks = [r for r in ranks if r <= max_rank]
    
plt.figure(figsize=(20, 10))
for i, r in enumerate(ranks):
    X_low = U[:, :r] @ np.diag(s[:r]) @ Vt[:r, :]
    first_frame = X_low[:, 0].reshape(h, w)
        
    plt.subplot(3, 5, i+1)
    plt.imshow(first_frame, cmap='gray')
    plt.title(f'Ранг = {r}')
    plt.axis('off')
plt.tight_layout()
plt.show()


Для фона нужен низкий ранг (1-3)